# 6.6. File I/O
D2L의 File I/O장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 왜 모델을 저장해야 하나?

딥러닝 모델을 학습하고 나면 학습된 결과를 나중에 다시 사용해야 한다.

예를 들어서

- 모델 학습을 끝낸 뒤 나중에 추론하기
- 서버에 모델 배포하기
- 긴 학습 도중 중간 상태 저장하기
- 학습이 중단되었을 때 다시 이어서 학습하기

등의 상황들이 있다. 모델이 학습됐다는 건 모델의 Weight와 Bias가 학습되었다는 것이고 따라서 학습된 파라미터를 파일로 저장해 두면 나중에 다시 불러와서 쓰면 된다.

며칠씩 학습하는 모델이라면 중간 결과를 저장하는 `checkpointing`이 중요하다.

## 2. Tensor 저장하기

PyTorch에서는 `torch.save()`로 데이터를 저장한다.

In [2]:
import torch

x = torch.arange(4)

print(x)

tensor([0, 1, 2, 3])


In [ ]:
torch.save(x, "x.pt") # save(저장할 데이터, 파일이름)

## 3. Tensor 다시 불러오기

저장된 건 `torch.load()`로 불러올 수 있다.

In [4]:
x_loaded = torch.load("x.pt")

print(x_loaded)

tensor([0, 1, 2, 3])


## 4. 여러 Tensor 저장하기

Tensor 하나만 저장할 수 있는 것은 아니다. 리스트나 딕셔너리 형태로 여러 데이터를 한 번에 저장할 수도 있다.

In [ ]:
x = torch.arange(4) # 리스트
y = torch.zeros(4)

torch.save([x, y], "xy.pt")

x_loaded, y_loaded = torch.load("xy.pt")

print(x_loaded)
print(y_loaded)

tensor([0, 1, 2, 3])
tensor([0., 0., 0., 0.])


In [ ]:
data = { # 딕셔너리
    "x": x,
    "y": y
}

torch.save(data, "data.pt")

loaded_data = torch.load("data.pt")

print(loaded_data)

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}


딕셔너리는

    이름 - Tensor

형태로 데이터를 관리할 수 있어서 신경망의 파라미터를 저장할 때 매우 유용하다.

## 5. 모델의 파라미터 확인하기

간단한 MLP 하나를 만들어 보자. 구조를 명확히 보기 위해 일반 `nn.Linear`를 사용하겠다.

In [7]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.hidden = nn.Linear(20, 256)
        self.relu = nn.ReLU()
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)

        return x

In [8]:
model = MLP()

X = torch.randn(2, 20)

Y = model(X)

print(Y.shape)

torch.Size([2, 10])


## 6. state_dict란?

In [9]:
model.state_dict()

OrderedDict([('hidden.weight',
              tensor([[ 0.1710,  0.1856, -0.0524,  ...,  0.0331, -0.1044,  0.0570],
                      [-0.1030, -0.0262, -0.0908,  ...,  0.1807,  0.0244, -0.0705],
                      [ 0.0601, -0.0606,  0.0941,  ..., -0.0039,  0.1750, -0.1589],
                      ...,
                      [-0.1045,  0.0582, -0.1842,  ...,  0.1768,  0.1539, -0.0388],
                      [ 0.0386, -0.1561, -0.1674,  ..., -0.1021,  0.1782, -0.0576],
                      [-0.0903,  0.0346, -0.0378,  ...,  0.1836, -0.1521,  0.0318]])),
             ('hidden.bias',
              tensor([-0.0465, -0.2186, -0.2173, -0.0293,  0.1600, -0.1474,  0.0728,  0.1289,
                      -0.1630,  0.0960,  0.0422, -0.2036, -0.0043, -0.2099,  0.1955,  0.1048,
                       0.0857,  0.1717,  0.1710,  0.2220,  0.0923, -0.1614,  0.2077,  0.0621,
                      -0.0881, -0.0832,  0.2092,  0.0497,  0.1625,  0.0174,  0.2066, -0.1636,
                       0.0533,

In [10]:
for name, param in model.state_dict().items():
    print(name, param.shape)

hidden.weight torch.Size([256, 20])
hidden.bias torch.Size([256])
output.weight torch.Size([10, 256])
output.bias torch.Size([10])


### state_dict

`state_dict()`는 모델이 가진 파라미터와 persistent buffer를 딕셔너리 형태로 모아 놓은 것이다.

예를 들어서

```text
hidden.weight
hidden.bias
output.weight
output.bias
```

같은 이름을 key로 가지고 실제 Weight와 Bias Tensor를 value로 가진다. 현재 모델이 학습한 Weight와 Bias 모음이라고 생각하면 된다.

```text
일반 딕셔너리

"x" - Tensor
"y" - Tensor

state_dict
"hidden.weight" - Tensor
"hidden.bias"   - Tensor
"output.weight" - Tensor
"output.bias"   - Tensor
```

## 7. 모델 저장하고 다시 불러오기

### 모델 저장

In [11]:
torch.save(model.state_dict(), "mlp.pth")

모델을 저장할 때는 보통 모델 자체보다는 `model.state_dict()`를 저장한다. 파일에 저장하는 것은 모델의 학습된 Weight와 Bias이다.

```text
Linear(20, 256)
ReLU()
Linear(256, 10)

이런 구조가 저장되지 않고

hidden.weight
hidden.bias
output.weight
output.bias

이런 값들이 저장된다.
```

모델을 복원하려면 먼저 같은 아키텍처를 코드로 생성하고 그 뒤 저장된 파라미터를 읽어야 한다.

### 불러오기

In [12]:
loaded_model = MLP()

state_dict = torch.load("mlp.pth")

loaded_model.load_state_dict(state_dict)

<All keys matched successfully>

In [14]:
loaded_model.eval() # 추론용이면

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=256, out_features=10, bias=True)
)

모델을 불러오는 과정은 두 단계가 있다.

1. 같은 모델 구조 생성
```py
loaded_model = MLP() # 처음에는 새로운 W, b가 있다.
```
2. 저장했던 파라미터 덮어쓰기
```py
loaded_model.load_state_dict( 
    torch.load("mlp.pth") # 새로운 모델의 W, b가 학습했던 값으로 교체된다.
)
```

## 8. 저장 전후 결과가 같은지 확인하기

In [15]:
model.eval()
loaded_model.eval()

with torch.no_grad():

    original_output = model(X)
    loaded_output = loaded_model(X)

print(torch.allclose(original_output, loaded_output))

True


같은 모델 구조에 같은 Weight와 Bias를 넣으면 동일한 입력에 대해 동일한 결과가 나온다.

## 9. 전체 흐름

In [ ]:
torch.save( # 저장
    model.state_dict(),
    "model.pth" # 모델 전체 저장보다는 모델의 상태(parameter/buffer)저장 
    # 생각하는게 좋다 
)

In [ ]:
model = MLP()

model.load_state_dict(
    torch.load("model.pth") # 불러오기
)

model.eval()

```text
model

↓ state_dict()

Weight / Bias 딕셔너리

↓ torch.save()

model.pth

↓ torch.load()

Weight / Bias 딕셔너리

↓ load_state_dict()

model
```

## 10. 오늘의 정리

- `torch.save()`는 Tensor나 딕셔너리 등의 데이터를 파일로 저장한다.
- `torch.load()`는 저장된 데이터를 다시 불러온다.
- 여러 Tensor를 리스트나 딕셔너리 형태로 한 번에 저장할 수 있다.
- `model.state_dict()`는 모델의 Weight와 Bias 등의 상태를 딕셔너리 형태로 가지고 있다.
- 일반적으로 PyTorch 모델은 `state_dict()`를 저장한다.
- 저장된 `state_dict`에는 모델 구조 자체가 들어있는 것이 아니다.
- 모델을 불러올 때는 먼저 동일한 모델 구조를 생성해야 한다.
- `load_state_dict()`를 사용하여 저장했던 Weight와 Bias를 모델에 넣는다.
- 추론할 때는 `model.eval()`을 호출한다.
- 같은 구조와 같은 파라미터를 가진 모델은 같은 입력에 대해 같은 결과를 출력한다.